In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking information about a fictional place called Lunapolis on the moon, including its capital, weather, population of cheese miners, and potential labor strikes.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as clear skies with temperatures ranging from a high of 120C to a low of -100C.\n- The population of cheese miners in Lunapolis is noted to be 100,000.\n- It is suggested that the cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='08fd6a93-c22a-4ea4-b621-4c3b05d47d0a'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='6642

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking information about a fictional place called Lunapolis on the moon, including its capital, weather, population of cheese miners, and potential labor strikes.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is described as clear skies with temperatures ranging from a high of 120C to a low of -100C.
- The population of cheese miners in Lunapolis is noted to be 100,000.
- It is suggested that the cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='f84be407-cebe-4908-90f1-1b62892297fa'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='368c5610-fa92-496f-a58d-e569895656dc', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='ed96c407-a4fe-42b1-aa43-183125d1946e'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='0e25ae49-4068-4bcf-ac73-b1a769e6cd50', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='a5224f38-7568-4d0b-b26b-4e546584ed9e'),
              AIMessage(content='I can’t measure the device’s temperature from here. If you tel

In [8]:
print(response["messages"][-1].content)

I can’t measure the device’s temperature from here. If you tell me the exact device type (e.g., laptop model, phone, desktop, etc.) and its OS, I can give model-specific steps to check temps. In the meantime, here are general tips for overheating or heat-related startup issues:

- If it feels hot, power it off and unplug it. Let it cool on a hard, flat surface before trying again.
- Ensure vents and fans aren’t blocked by dust, a carpet, or a soft surface. Clean dust from vents if you can safely do so.
- Use a cooling solution (for laptops): a cooling pad or stand to improve airflow; avoid using on beds or couches.
- Check the power adapter and cable. If the charger or brick gets unusually hot, try a different compatible charger and cable.
- If you can boot, monitor temps with software (examples):
  - Windows: HWMonitor, Core Temp, or HWInfo
  - macOS: Macs Fan Control or iStat Menus
  - Android: various temperature/diagnostic apps (requires boot and OS access)
  - iPhone: temperature 